# D3 Tree SQL Debug Notebook

This notebook inspects the output of the `get_tree_data_for_family` SQL query used for the D3.js tree in the Flask webapp.

**Purpose:**
- Connects to the business_context.sqlite database
- Runs the D3 tree query for a selected job family
- Displays the number of nodes, their types, and parent-child relationships
- Helps diagnose why the tree is overpopulated or incorrectly structured

**Expected Structure:**
- Job Family Root (1 node)
- Individual Jobs (maybe 10-50 nodes per family)
- Similar Jobs (maybe 5-20 per job, with similarity >= 0.6)


In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Path to the SQLite database
db_path = Path(r'C:/Users/kipjo/OneDrive/Documents/GitHub/skill-similarity-engine/models/2025-Q2/business_context.sqlite')
print(f'Checking database: {db_path}')
assert db_path.exists(), f'Database not found: {db_path}'
print('✅ Database found!')

# Connect to database
conn = sqlite3.connect(str(db_path))
print('✅ Connected to database')


Checking database: C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\models\2025-Q2\business_context.sqlite
✅ Database found!
✅ Connected to database


In [2]:
# First, let's see what job families are available
families_query = "SELECT JobFamily, COUNT(*) as job_count FROM jobs GROUP BY JobFamily ORDER BY job_count DESC"
families_df = pd.read_sql_query(families_query, conn)
print('Available job families:')
print(families_df)


Available job families:
                  JobFamily  job_count
0        Banking Operations        101
1          Data & Analytics         99
2      Finance & Accounting         98
3           Human Resources         94
4      Executive Leadership         88
5  Customer Service & Sales         81
6         Risk & Compliance         78
7  Technology & Engineering         76


In [3]:
# Read the D3 tree SQL query from the .sql file
sql_file = Path('../src/skill_similarity_engine/webapp/sql/d3_visualization.sql')
print(f'Reading SQL from: {sql_file}')

with open(sql_file, 'r', encoding='utf-8') as f:
    sql_content = f.read()

# Extract the get_tree_data_for_family query
lines = sql_content.split('\n')
query_lines = []
in_query = False
for line in lines:
    if '-- query_name: get_tree_data_for_family' in line:
        in_query = True
        continue
    elif in_query and line.strip().startswith('-- query_name:') and 'get_tree_data_for_family' not in line:
        break
    elif in_query and not line.strip().startswith('--'):
        query_lines.append(line)

d3_tree_query = '\n'.join(query_lines).strip()
print('--- D3 Tree Query (first 500 chars) ---')
print(d3_tree_query[:500])
print('\n... (truncated)')
print(f'\nQuery length: {len(d3_tree_query)} characters')


Reading SQL from: ..\src\skill_similarity_engine\webapp\sql\d3_visualization.sql
--- D3 Tree Query (first 500 chars) ---
SELECT 
    'family' as node_type,
    j.JobFamily as id,
    j.JobFamily as name,
    NULL as parent_id,
    COUNT(*) as children_count,
    0 as similarity_score,
    'Job Family' as category
FROM jobs j
WHERE j.JobFamily = ?
GROUP BY j.JobFamily

UNION ALL

SELECT 
    'job' as node_type,
    j.JobProfileID as id,
    j.JobProfile as name,
    j.JobFamily as parent_id,
    COUNT(DISTINCT js.job_to) as children_count,
    0 as similarity_score,
    COALESCE(j.JobFamilyGroup, 'Unknown') as cate

... (truncated)

Query length: 1756 characters


In [4]:
# Choose a job family to test (pick one with a moderate number of jobs)
job_family = 'Executive Leadership'  # Change this to test different families

print(f'🔍 Testing job family: {job_family}')
print('Running D3 tree query...')

# Run the query with the job family parameter (used 3 times)
try:
    df = pd.read_sql_query(d3_tree_query, conn, params=[job_family, job_family, job_family])
    print(f'✅ Query completed successfully')
    print(f'📊 Total rows returned: {len(df)}')
    print(f'📋 Columns: {list(df.columns)}')
except Exception as e:
    print(f'❌ Query failed: {e}')
    df = None


🔍 Testing job family: Executive Leadership
Running D3 tree query...
✅ Query completed successfully
📊 Total rows returned: 381
📋 Columns: ['node_type', 'id', 'name', 'parent_id', 'children_count', 'similarity_score', 'category']


In [5]:
# Analyze the results if query succeeded
if df is not None:
    print('=== NODE TYPE ANALYSIS ===')
    node_counts = df['node_type'].value_counts()
    print(node_counts)
    
    print('\n=== SAMPLE DATA ===')
    print(df[['id', 'name', 'node_type', 'parent_id', 'children_count', 'similarity_score']].head(15))


=== NODE TYPE ANALYSIS ===
node_type
similar_job    292
job             88
family           1
Name: count, dtype: int64

=== SAMPLE DATA ===
                      id                            name node_type  \
0   Executive Leadership            Executive Leadership    family   
1                R0222.2    Analyst - Chief Risk Officer       job   
2                R0099.2         Analyst - Division Head       job   
3                R0266.3         Analyst - Division Head       job   
4                R0002.1    Analyst - Executive Director       job   
5                R0081.6    Analyst - Executive Director       job   
6                R0187.2       Analyst - General Manager       job   
7                R0223.2       Analyst - General Manager       job   
8                R0285.5       Analyst - General Manager       job   
9                R0449.3       Analyst - General Manager       job   
10               R0025.2         Analyst - Regional Head       job   
11               R0

In [6]:
# Check if the problem is in the SQL query itself
print('=== SQL QUERY INVESTIGATION ===')
print('\nTesting each part of the UNION query separately...')

# Part 1: Family node
family_query = """
SELECT 
    'family' as node_type,
    j.JobFamily as id,
    j.JobFamily as name,
    NULL as parent_id,
    COUNT(*) as children_count,
    0 as similarity_score,
    'Job Family' as category
FROM jobs j
WHERE j.JobFamily = ?
GROUP BY j.JobFamily
"""

family_df = pd.read_sql_query(family_query, conn, params=[job_family])
print(f'Part 1 (Family): {len(family_df)} rows')
print(family_df)

# Part 2: Job nodes
job_query = """
SELECT 
    'job' as node_type,
    j.JobProfileID as id,
    j.JobProfile as name,
    j.JobFamily as parent_id,
    COUNT(DISTINCT js.job_to) as children_count,
    0 as similarity_score,
    COALESCE(j.JobFamilyGroup, 'Unknown') as category
FROM jobs j
LEFT JOIN job_similarities js ON j.JobProfileID = js.job_from AND js.similarity_score >= 0.6
WHERE j.JobFamily = ?
GROUP BY j.JobProfileID, j.JobProfile, j.JobFamily, j.JobFamilyGroup
"""

jobs_df = pd.read_sql_query(job_query, conn, params=[job_family])
print(f'\nPart 2 (Jobs): {len(jobs_df)} rows')
print(jobs_df.head())
print(f'Children count range: {jobs_df["children_count"].min()} - {jobs_df["children_count"].max()}')

# Part 3: Similar job nodes
similar_query = """
SELECT 
    'similar_job' as node_type,
    'sim_' || js.job_from || '_' || js.job_to as id,
    similar_job.JobProfile as name,
    js.job_from as parent_id,
    0 as children_count,
    js.similarity_score,
    CASE 
        WHEN js.similarity_score >= 0.8 THEN 'High'
        WHEN js.similarity_score >= 0.6 THEN 'Medium'
        ELSE 'Low'
    END as category
FROM job_similarities js
JOIN jobs source_job ON js.job_from = source_job.JobProfileID
JOIN jobs similar_job ON js.job_to = similar_job.JobProfileID
WHERE source_job.JobFamily = ?
    AND js.similarity_score >= 0.6
"""

similar_df = pd.read_sql_query(similar_query, conn, params=[job_family])
print(f'\nPart 3 (Similar Jobs): {len(similar_df)} rows')
print(similar_df.head())
print(f'Similarity range: {similar_df["similarity_score"].min():.3f} - {similar_df["similarity_score"].max():.3f}')

print(f'\n=== SUMMARY ===')
print(f'Family nodes: {len(family_df)}')
print(f'Job nodes: {len(jobs_df)}')
print(f'Similar job nodes: {len(similar_df)}')
print(f'Total: {len(family_df) + len(jobs_df) + len(similar_df)}')
print(f'Original query result: {len(df) if df is not None else "Failed"}')

=== SQL QUERY INVESTIGATION ===

Testing each part of the UNION query separately...
Part 1 (Family): 1 rows
  node_type                    id                  name parent_id  \
0    family  Executive Leadership  Executive Leadership      None   

   children_count  similarity_score    category  
0              88                 0  Job Family  

Part 2 (Jobs): 88 rows
  node_type       id                                     name  \
0       job  R0002.0      Senior Associate Executive Director   
1       job  R0002.1             Analyst - Executive Director   
2       job  R0003.2            Graduate - Executive Director   
3       job  R0007.1  Executive Director - Executive Director   
4       job  R0009.1                Associate General Manager   

              parent_id  children_count  similarity_score  \
0  Executive Leadership               7                 0   
1  Executive Leadership               7                 0   
2  Executive Leadership               1                

In [7]:
# Test with different job families
print('=== TESTING MULTIPLE JOB FAMILIES ===')

families_to_test = ['Executive Leadership', 'Data & Analytics', 'Banking Operations', 'Technology']

for test_family in families_to_test:
    if test_family in families_df['JobFamily'].values:
        try:
            test_df = pd.read_sql_query(d3_tree_query, conn, params=[test_family, test_family, test_family])
            node_counts = test_df['node_type'].value_counts()
            print(f'\n{test_family}:')
            print(f'  Total nodes: {len(test_df)}')
            print(f'  Family: {node_counts.get("family", 0)}')
            print(f'  Jobs: {node_counts.get("job", 0)}')
            print(f'  Similar jobs: {node_counts.get("similar_job", 0)}')
        except Exception as e:
            print(f'\n{test_family}: ERROR - {e}')
    else:
        print(f'\n{test_family}: Not found in database')




=== TESTING MULTIPLE JOB FAMILIES ===

Executive Leadership:
  Total nodes: 381
  Family: 1
  Jobs: 88
  Similar jobs: 292

Data & Analytics:
  Total nodes: 430
  Family: 1
  Jobs: 99
  Similar jobs: 330

Banking Operations:
  Total nodes: 439
  Family: 1
  Jobs: 101
  Similar jobs: 337

Technology: Not found in database


In [8]:
# Test the IMPROVED query with limited similar jobs
print('=== TESTING IMPROVED QUERY (Top 5 Similar Jobs + 0.7 Threshold) ===')

# Re-read the updated SQL query
with open(sql_file, 'r', encoding='utf-8') as f:
    updated_sql_content = f.read()

# Extract the updated get_tree_data_for_family query
lines = updated_sql_content.split('\n')
query_lines = []
in_query = False
for line in lines:
    if '-- query_name: get_tree_data_for_family' in line:
        in_query = True
        continue
    elif in_query and line.strip().startswith('-- query_name:') and 'get_tree_data_for_family' not in line:
        break
    elif in_query and not line.strip().startswith('--'):
        query_lines.append(line)

updated_d3_tree_query = '\n'.join(query_lines).strip()

print('--- Updated D3 Tree Query (first 400 chars) ---')
print(updated_d3_tree_query[:400])
print('...')

# Test updated query
families_to_test = ['Executive Leadership', 'Data & Analytics', 'Banking Operations']

for test_family in families_to_test:
    try:
        test_df = pd.read_sql_query(updated_d3_tree_query, conn, params=[test_family, test_family, test_family])
        node_counts = test_df['node_type'].value_counts()
        print(f'\n✅ {test_family}:')
        print(f'  Total nodes: {len(test_df)} (was {954 if test_family == "Executive Leadership" else "N/A"})')
        print(f'  Family: {node_counts.get("family", 0)}')
        print(f'  Jobs: {node_counts.get("job", 0)}')
        print(f'  Similar jobs: {node_counts.get("similar_job", 0)} (was 865+ before)')
        
        # Calculate average similar jobs per job
        if node_counts.get("job", 0) > 0 and node_counts.get("similar_job", 0) > 0:
            avg_similar = node_counts.get("similar_job", 0) / node_counts.get("job", 0)
            print(f'  Avg similar jobs per job: {avg_similar:.1f} (should be ≤5)')
    except Exception as e:
        print(f'\n❌ {test_family}: Error - {e}')

print('\n🎯 The tree should now have manageable sizes!')


=== TESTING IMPROVED QUERY (Top 5 Similar Jobs + 0.7 Threshold) ===
--- Updated D3 Tree Query (first 400 chars) ---
SELECT 
    'family' as node_type,
    j.JobFamily as id,
    j.JobFamily as name,
    NULL as parent_id,
    COUNT(*) as children_count,
    0 as similarity_score,
    'Job Family' as category
FROM jobs j
WHERE j.JobFamily = ?
GROUP BY j.JobFamily

UNION ALL

SELECT 
    'job' as node_type,
    j.JobProfileID as id,
    j.JobProfile as name,
    j.JobFamily as parent_id,
    COUNT(DISTINCT js.job
...

✅ Executive Leadership:
  Total nodes: 381 (was 954)
  Family: 1
  Jobs: 88
  Similar jobs: 292 (was 865+ before)
  Avg similar jobs per job: 3.3 (should be ≤5)

✅ Data & Analytics:
  Total nodes: 430 (was N/A)
  Family: 1
  Jobs: 99
  Similar jobs: 330 (was 865+ before)
  Avg similar jobs per job: 3.3 (should be ≤5)

✅ Banking Operations:
  Total nodes: 439 (was N/A)
  Family: 1
  Jobs: 101
  Similar jobs: 337 (was 865+ before)
  Avg similar jobs per job: 3.3 (should be ≤5

In [9]:
conn.close()
print('\n✅ Database connection closed')


✅ Database connection closed
